# 05. YOLO26n — Detection과 사람 Pose / 90분

**선행:** 01–04의 좌표 추종·판단·시간 검증, 06.OpenCV 영상 좌표

**이 노트북의 결과:** `yolo26n.pt`로 검출 상자를 확인하고 `yolo26n-pose.pt`로 사람별 17개 관절을 추출합니다. 검출 선택 TODO와 가장 큰 사람 선택 TODO를 완성하고, 다음 06에서 이미지 없이 LLM에 보낼 사람 전체 좌표를 준비합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

뒤의 LLM이 사용할 좌표가 어디에서 나오는지 이해하는 실습입니다. detection으로 box·종류·점수를 읽고 pose로 사람의 17개 관절을 이름 있는 좌표로 표현합니다. 두 모델을 반드시 직렬 연결하는 것이 아니라 pose 결과에서 직접 사람을 선택합니다.

### 전체 과정에서 내 코드가 하는 일

```text
detection 결과 계약 TODO
  ↓
실제 box·JSON
  ↓
pose 관절
  ↓
사람 신뢰도 검사
  ↓
최대 면적 선택 TODO
  ↓
전체 사람 JSON
```

검출 필터와 가장 큰 사람 선택의 두 TODO를 작성합니다. 사람 confidence와 코 confidence를 구분하고, 같은 면적이면 먼저 나온 후보를 유지하며 선택 결과는 깊게 복사합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 가장 큰 사람의 코가 가려졌으면 다른 사람의 코를 쓰나요?

**설명:** 현재 계약에서는 사람 선택과 목표 관절 검사가 분리되어 있습니다. 먼저 선택된 사람의 코가 유효한지 확인하고 아니면 STOP합니다. 임의로 다른 사람의 코를 빌리면 선택한 사람 전체를 넘긴다는 계약이 달라집니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 슬라이드와 함께 진행하기

시작 안내는 [4일차 슬라이드 49쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s046) · [배포 PDF 49쪽](../슬라이드/4일차.pdf#page=49)입니다. 노트북의 설명·TODO·검증·실제 실행을 진행한 뒤, [4일차 슬라이드 50쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s006) · [배포 PDF 50쪽](../슬라이드/4일차.pdf#page=50)로 돌아가 50–63쪽에서 결과를 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.


## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


## 시간 지도

Detection·박스 10분 → 검출 TODO·검증 20분 → 실제 detection 10분 → Pose·17개 관절 15분 → 가장 큰 사람 TODO·검증 20분 → 실제 pose·저장 15분


## 필요한 앞 내용 — 위치와 핵심을 함께 확인

[06.OpenCV/01_image_video_io.ipynb의 이미지 배열 실습](../06.OpenCV/01_image_video_io.ipynb)에 영상 shape가 있습니다. 여기서 필요한 내용은 `image.shape=(높이,폭,채널)`, 픽셀 접근은 `image[y,x]`, x는 오른쪽·y는 아래라는 점입니다. 검출 사각형은 픽셀 경계 `(x1,y1,x2,y2)`이고 모델 class ID는 종류 이름으로 바꿉니다. dict는 한 검출의 필드, list는 여러 검출을 담습니다. 이 노트북의 `DETECTIONS`가 입력을 직접 만들므로 이전 산출물은 필요하지 않습니다.

실제 실행에 필요한 배포 자산은 `/workspace/Data/models/yolo26n.pt`, `/workspace/Data/models/yolo26n-pose.pt`, `/workspace/Data/images/road.png`, `/workspace/Data/images/07.Mediapipe/person_open_palm.png`입니다. 파일 fixture 뒤 USB 카메라 실습은 로컬 환경에서 한 번만 실행합니다.

In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. Detection 출력은 명령이 아니다

<!-- procedure:lesson-34-15e84c:start -->
![검출에서 사람 전체 좌표까지](../교재그림/절차/lesson-34.svg)

같은 원본에 detection과 pose를 각각 실행합니다. 06에는 pose에서 선택한 한 사람의 전체 관절을 전달합니다.
<!-- procedure:lesson-34-15e84c:end -->


```text
image [H,W,3] → model → boxes [N,4], scores [N], class IDs [N] → 이름 있는 detection N개
```

`N=0`도 정상 미검출 결과일 수 있습니다. `person`을 찾았다는 관측은 전진 명령이 아닙니다. 이 자료의 box는 pixel `xyxy=[x1,y1,x2,y2]`이고 `0≤x1<x2≤width`, `0≤y1<y2≤height`입니다.

Confidence는 0–1 model score이며 현실에서 참일 확률로 자동 보정되지 않습니다. Threshold 0.70에는 정확히 0.70을 포함하도록 계약합니다. Bool·문자열·NaN·Inf를 숫자로 조용히 받아들이지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
DETECTIONS = [
    {"label": "person", "confidence": 0.90, "bbox_xyxy": [10, 10, 30, 40], "frame_id": "f1", "captured_ms": 100},
    {"label": "box", "confidence": 0.50, "bbox_xyxy": [0, 0, 5, 5], "frame_id": "f1", "captured_ms": 100},
    {"label": "person", "confidence": 0.70, "bbox_xyxy": [40, 10, 60, 40], "frame_id": "f1", "captured_ms": 100},
]
print("입력:", [(d["label"], d["confidence"]) for d in DETECTIONS])
print("person, threshold 0.70이면 몇 개이며 순서는 어떻게 될지 예상하세요.")


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. TODO — Detection 선택

[exercises/detection_contract.py](exercises/detection_contract.py)의 `select_detections`를 구현합니다. 모든 record의 정확한 schema와 좌표를 먼저 검사하고 class/threshold를 적용합니다. Confidence 내림차순, label/box 순으로 결정적으로 정렬하며 입력 list·dict를 바꾸지 않습니다. Malformed detection을 조용히 버리면 센서 오류와 미검출을 구분하지 못하므로 예외로 거부합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

#### 1) 함수에 들어오는 데이터 — schema

**schema(스키마)** 는 데이터의 필드 이름·자료형·값의 조건을 정한 규칙입니다. `records`에는 **검출 한 개를 dict 하나로 표현한 목록** 이 들어옵니다. 앞의 예제 셀의 `DETECTIONS`가 바로 이 입력입니다. 이미지 배열이나 YOLO 결과 객체를 직접 받는 함수가 아닙니다.

실제 경로의 [scripts/live_yolo.py](scripts/live_yolo.py)는 YOLO의 `boxes.xyxy`·`boxes.conf`·`boxes.cls`를 읽고, class ID를 `result.names`로 이름에 대응시킨 뒤 같은 구조를 만듭니다. ID는 식별 번호, bbox는 bounding box(객체를 둘러싼 사각형), conf는 confidence(검출 점수), cls는 class(객체 종류)의 약어입니다.

```python
# records 안의 검출 한 개: 원본 영상 폭 100, 높이 80 픽셀인 예시
{
    "label": "person",
    "confidence": 0.90,
    "bbox_xyxy": [10, 10, 30, 40],
    "frame_id": "f1",
    "captured_ms": 100,
}
```

| 필드 | 들어오는 값과 의미 | 검사할 조건 |
| --- | --- | --- |
| `label` | `"person"`: 객체 종류의 이름 | 비어 있지 않은 문자열 |
| `confidence` | `0.90`: 검출 점수 | 0 이상 1 이하의 유한한 정수·실수 |
| `bbox_xyxy` | `[10, 10, 30, 40]`: 사각형의 두 꼭짓점 | 길이 4인 list 또는 tuple, 유한한 숫자 네 개, 아래 좌표 조건 만족 |
| `frame_id` | `"f1"`: 관측한 프레임의 식별 문자열 | 비어 있지 않은 문자열 |
| `captured_ms` | `100`: 캡처 시각을 밀리초 단위로 표현한 값 | 0 이상의 정수; 처리 소요 시간이 아님 |

`ms`는 millisecond(밀리초, 1/1000초)입니다. `100`은 검증용 예시 시각이며 현재 시각을 뜻하지 않습니다. **위 다섯 key가 정확히 있어야 합니다.** 누락되거나 다른 key가 추가되면 오류입니다. 숫자 필드에 `True`·`False`, 문자열 `"0.9"`, `NaN`(숫자가 아님), `Inf`(무한대)를 허용하지 않습니다.

#### 2) 좌표를 읽는 방법 — xyxy와 영상 크기

**좌표는 픽셀 단위이며, 원점 `(0, 0)`은 영상의 왼쪽 위입니다.** x는 오른쪽으로, y는 아래로 증가합니다. `[x1, y1, x2, y2]`에서 `(x1, y1)`은 사각형 왼쪽 위, `(x2, y2)`는 오른쪽 아래 경계입니다. 0–1로 정규화한 비율이나 `[x, y, 폭, 높이]` 형식으로 해석하지 않습니다. 좌표는 소수여도 되므로 정수로 반올림할 필요가 없습니다.

```text
(0, 0) ─────────────────────→ x (가로)
  │    (10, 10) ┌────────┐
  │             │ person │  높이 = 40 - 10 = 30
  │             └────────┘ (30, 40)
  ↓              폭 = 30 - 10 = 20
 y (세로)
```

이 예시의 영상 전체 크기는 `image_width=100`, `image_height=80`입니다. 다음 두 조건을 모두 만족해야 합니다.

```python
0 <= x1 < x2 <= image_width
0 <= y1 < y2 <= image_height
```

따라서 `[0, 0, 100, 80]`은 정상입니다. 오른쪽·아래 **경계 좌표** 는 폭·높이와 같아도 되지만, 배열의 마지막 픽셀 인덱스는 각각 99·79입니다. `[10, 10, 101, 40]`은 영상 밖, `[30, 10, 10, 40]`은 좌우 역전, `[10, 10, 10, 40]`은 폭 0이므로 오류입니다. TODO에서는 잘못된 좌표를 잘라 맞추거나 자동 수정하지 않고 예외로 거부합니다.

#### 3) 나머지 입력 인자와 기대 출력

| 인자 | 이 노트북의 값 | 의미와 조건 |
| --- | --- | --- |
| `allowed_labels` | `{"person"}` | 허용할 이름의 모음. 비어 있지 않은 문자열을 한 개 이상 포함; 문자열 하나 `"person"`을 그대로 넘기지 않음 |
| `min_confidence` | `0.70` | 포함할 점수의 하한. 0 이상 1 이하 유한수, bool 제외. **하한과 같은 점수도 포함** |
| `image_width` | `100` | 영상 전체의 가로 픽셀 수. bool이 아닌 양의 정수 |
| `image_height` | `80` | 영상 전체의 세로 픽셀 수. bool이 아닌 양의 정수 |

위의 `DETECTIONS`에는 `person/0.90`, `box/0.50`, `person/0.70` 세 검출이 있습니다. `allowed_labels={"person"}`, `min_confidence=0.70`으로 호출하면 다음 **새 dict의 목록** 을 반환해야 합니다.

```python
[
    {"label": "person", "confidence": 0.90, "bbox_xyxy": [10.0, 10.0, 30.0, 40.0], "frame_id": "f1", "captured_ms": 100},
    {"label": "person", "confidence": 0.70, "bbox_xyxy": [40.0, 10.0, 60.0, 40.0], "frame_id": "f1", "captured_ms": 100},
]
```

출력도 같은 다섯 필드를 유지하고 `bbox_xyxy`는 list로 반환합니다. 입력 list·dict·좌표를 변경하지 않습니다. 정상 입력 중 선택된 검출이 없거나 입력 목록이 비어 있으면 `[]`를 반환합니다. 옵션이나 검출 데이터가 잘못되었을 때의 예외와 정상 미검출 `[]`를 구분하세요.

#### 4) TODO 작성 흐름 힌트

1. 입력 목록과 옵션을 검사합니다. `records`는 list·tuple 같은 sequence이며 문자열은 허용하지 않습니다.
2. **모든 record** 의 key·자료형·숫자 범위·좌표·시각을 검사합니다. 제외할 `box`의 좌표가 잘못되었어도 예외를 내야 하므로 필터보다 먼저 검사합니다.
3. 정상 record 중 label이 허용 목록에 있고 confidence가 하한 이상인 것만 고릅니다.
4. confidence 내림차순으로 정렬합니다. 점수가 같으면 label 오름차순, label까지 같으면 `(x1, y1, x2, y2)`를 앞 좌표부터 비교하는 오름차순입니다. 정렬 key의 힌트는 `(-confidence, label, bbox)`입니다.
5. 새 dict와 새 bbox list로 결과를 만들어 반환합니다. 기존 함수 이름과 인자는 유지하고 필요한 검사 보조 함수는 같은 파일에 추가해도 됩니다.

**손으로 확인:** 하한을 `0.90`으로 올리면 첫 person만 남고, `0.91`이면 `[]`입니다. 높은 점수라도 좌표가 잘못된 검출은 통과시키지 않습니다.

<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_vla_detection", "exercises/detection_contract.py")
before = json.dumps(DETECTIONS, sort_keys=True)
selected = student.select_detections(
    DETECTIONS, allowed_labels={"person"}, min_confidence=0.70,
    image_width=100, image_height=80,
)
assert [d["confidence"] for d in selected] == [0.90, 0.70]
assert all(set(d) == {"label", "confidence", "bbox_xyxy", "frame_id", "captured_ms"} for d in selected)
assert all(d["label"] == "person" for d in selected)
assert json.dumps(DETECTIONS, sort_keys=True) == before

# 허용 label에서도 threshold 직전은 제외하고 정확히 같은 값부터 포함합니다.
# box처럼 label 때문에 제외되는 입력만 검사하면 confidence 필터 누락을 놓칩니다.
boundary_records = [
    {**DETECTIONS[0], "confidence": confidence}
    for confidence in (0.699, 0.700, 0.701)
]
boundary_before = json.dumps(boundary_records, sort_keys=True)
boundary_result = student.select_detections(
    boundary_records, allowed_labels={"person"}, min_confidence=0.700,
    image_width=100, image_height=80,
)
assert [d["confidence"] for d in boundary_result] == [0.701, 0.700], "confidence 경계의 제외/포함을 확인하세요"
assert json.dumps(boundary_records, sort_keys=True) == boundary_before

# 같은 confidence에서는 먼저 label, 같은 label에서는 box 좌표가 결과를 결정해야 합니다.
ties = [
    {"label": "person", "confidence": 0.8, "bbox_xyxy": [30, 10, 40, 20], "frame_id": "f2", "captured_ms": 200},
    {"label": "box", "confidence": 0.8, "bbox_xyxy": [20, 10, 30, 20], "frame_id": "f2", "captured_ms": 200},
    {"label": "person", "confidence": 0.8, "bbox_xyxy": [10, 10, 20, 20], "frame_id": "f2", "captured_ms": 200},
]
tie_result = student.select_detections(
    ties, allowed_labels={"person", "box"}, min_confidence=0.8,
    image_width=100, image_height=80,
)
assert [(d["label"], d["bbox_xyxy"]) for d in tie_result] == [
    ("box", [20, 10, 30, 20]),
    ("person", [10, 10, 20, 20]),
    ("person", [30, 10, 40, 20]),
]
assert student.select_detections(
    [], allowed_labels={"person"}, min_confidence=0.5,
    image_width=100, image_height=80,
) == []

def expect_rejected(records, **overrides):
    options = dict(
        allowed_labels={"person"}, min_confidence=0.5,
        image_width=100, image_height=80,
    )
    options.update(overrides)
    try:
        student.select_detections(records, **options)
    except (TypeError, ValueError):
        return
    raise AssertionError(f"거부해야 하는 detection/config입니다: {records!r}, {overrides!r}")

valid = DETECTIONS[0]
bad_records = [
    {**valid, "extra": "금지된 필드"},
    {key: value for key, value in valid.items() if key != "label"},
    {**valid, "label": ""},
    {**valid, "confidence": True},
    {**valid, "confidence": "0.9"},
    {**valid, "confidence": float("nan")},
    {**valid, "confidence": float("inf")},
    {**valid, "confidence": 1.1},
    {**valid, "bbox_xyxy": [10, 10, 20]},
    {**valid, "bbox_xyxy": [10, 10, 10, 20]},
    {**valid, "bbox_xyxy": [10, 10, 101, 20]},
    {**valid, "bbox_xyxy": [True, 10, 20, 20]},
    {**valid, "frame_id": ""},
    {**valid, "captured_ms": True},
    {**valid, "captured_ms": -1},
]
for bad in bad_records:
    expect_rejected([bad])
for overrides in (
    {"allowed_labels": "person"}, {"allowed_labels": set()}, {"allowed_labels": {1}},
    {"min_confidence": True}, {"min_confidence": -0.1}, {"min_confidence": float("nan")},
    {"image_width": 0}, {"image_height": True},
):
    expect_rejected([valid], **overrides)
print("PASS: exact schema/type/range, inclusive threshold, deterministic sort, input preservation")


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 참고 정답

강사가 안내하는 시점에 아래 접힌 정답 셀을 여세요. 셀은 `solutions/`의 독립적인 참고 구현이 같은 계약을 통과하는지 확인합니다. 바로 아래 정답 해설·접힌 코드에 입력 검사·경계 처리·반환값을 다시 제공합니다. 실제 카메라·ROS 공통 구현은 별도 runtime이며 참고 정답 실행과 학생 TODO 검증은 각각 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_vla_detection", "solutions/detection_contract.py")
assert len(reference.select_detections(
    DETECTIONS, allowed_labels={"person"}, min_confidence=0.70,
    image_width=100, image_height=80,
)) == 2
print("PASS: 참고 정답 검증")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/detection_contract.py](solutions/detection_contract.py)의 `select_detections`입니다. schema·좌표를 검사한 뒤 허용 label과 confidence 기준으로 고르고 정렬합니다.

xyxy는 왼쪽·위·오른쪽·아래 좌표이며 width/height와 비교해야 합니다. 범위 밖이나 뒤집힌 상자를 높은 confidence만 보고 통과시키지 않습니다. 동일 confidence에서는 정답의 후속 정렬 기준도 읽어 반복 결과 순서를 설명합니다.

**해석:** YOLO confidence는 제어 승인이나 실제 정답 확률을 자동으로 뜻하지 않습니다. 실제 손 추종 위치는 MediaPipe 관절 좌표에서 얻고 YOLO는 별도의 관측 표현 학습입니다.

**확인 질문:** 가장 높은 confidence 상자가 허용하지 않은 class라면 선택해야 하나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/detection_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 모든 검출의 필드·좌표·시각을 검사한 뒤 label과 confidence로 필터링합니다.
점수 내림차순, 이름·좌표 순으로 정렬한 새 dict 목록을 반환합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

import math
from dataclasses import asdict, dataclass
from typing import Any, Iterable, Mapping, Sequence

def finite_number(value: Any, name: str) -> float:
    """bool, 문자열, NaN, Inf를 숫자로 조용히 받아들이지 않는다."""
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"{name}은 bool이 아닌 숫자여야 합니다")
    try:
        number = float(value)
    except OverflowError as error:
        raise ValueError(f"{name}은 표현 가능한 유한수여야 합니다") from error
    if not math.isfinite(number):
        raise ValueError(f"{name}은 유한한 숫자여야 합니다")
    return number


def non_negative_int(value: Any, name: str) -> int:
    if isinstance(value, bool) or not isinstance(value, int) or value < 0:
        raise ValueError(f"{name}은 0 이상의 정수여야 합니다")
    return value


@dataclass(frozen=True)
class Detection:
    label: str
    confidence: float
    bbox_xyxy: tuple[float, float, float, float]
    frame_id: str
    captured_ms: int

    def as_dict(self) -> dict[str, Any]:
        value = asdict(self)
        value["bbox_xyxy"] = list(self.bbox_xyxy)
        return value


def _detection_from_mapping(record: Mapping[str, Any], width: int, height: int) -> Detection:
    required = {"label", "confidence", "bbox_xyxy", "frame_id", "captured_ms"}
    if not isinstance(record, Mapping) or set(record) != required:
        raise ValueError(f"detection 필드는 정확히 {sorted(required)}여야 합니다")
    label = record["label"]
    frame_id = record["frame_id"]
    if not isinstance(label, str) or not label:
        raise ValueError("label은 비어 있지 않은 문자열이어야 합니다")
    if not isinstance(frame_id, str) or not frame_id:
        raise ValueError("frame_id는 비어 있지 않은 문자열이어야 합니다")
    confidence = finite_number(record["confidence"], "confidence")
    if not 0.0 <= confidence <= 1.0:
        raise ValueError("confidence는 0과 1 사이여야 합니다")
    bbox = record["bbox_xyxy"]
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        raise ValueError("bbox_xyxy는 [x1,y1,x2,y2] 네 숫자여야 합니다")
    x1, y1, x2, y2 = (finite_number(v, "bbox coordinate") for v in bbox)
    if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
        raise ValueError("bbox는 영상 안에 있고 양의 넓이를 가져야 합니다")
    captured_ms = non_negative_int(record["captured_ms"], "captured_ms")
    return Detection(label, confidence, (x1, y1, x2, y2), frame_id, captured_ms)


def select_detections(
    records: Sequence[Mapping[str, Any]],
    *,
    allowed_labels: Iterable[str],
    min_confidence: float,
    image_width: int,
    image_height: int,
) -> list[dict[str, Any]]:
    """검출 schema를 검사하고 허용 class/threshold를 통과한 결과를 정렬한다."""
    if not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise TypeError("records는 detection mapping의 sequence여야 합니다")
    width = non_negative_int(image_width, "image_width")
    height = non_negative_int(image_height, "image_height")
    if width == 0 or height == 0:
        raise ValueError("영상 크기는 양수여야 합니다")
    if isinstance(allowed_labels, (str, bytes)):
        raise TypeError("allowed_labels는 문자열 하나가 아니라 label iterable이어야 합니다")
    labels = frozenset(allowed_labels)
    if not labels or not all(isinstance(v, str) and v for v in labels):
        raise ValueError("allowed_labels에는 한 개 이상의 문자열이 필요합니다")
    threshold = finite_number(min_confidence, "min_confidence")
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("min_confidence는 0과 1 사이여야 합니다")
    # 필터에서 제외될 record도 먼저 검사해야 손상된 센서 입력을 놓치지 않습니다.
    parsed = [_detection_from_mapping(record, width, height) for record in records]
    selected = [item for item in parsed if item.label in labels and item.confidence >= threshold]
    selected.sort(key=lambda item: (-item.confidence, item.label, item.bbox_xyxy))
    return [item.as_dict() for item in selected]
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
selected = student.select_detections(
    DETECTIONS, allowed_labels={"person"}, min_confidence=0.70,
    image_width=100, image_height=80,
)
output = ROOT / "Data/outputs/09.Pseudo-VLA/detection-fixture.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps({"frame_id": "f1", "detections": selected}, ensure_ascii=False, indent=2) + "\n")
print("saved:", output)


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 실제 YOLO 경로

```bash
bash
python /workspace/09.Pseudo-VLA/scripts/live_yolo.py   --task detect   --model /workspace/Data/models/yolo26n.pt   --source /workspace/Data/images/road.png
```

사전 배포 model/image, Ultralytics version과 CUDA를 확인합니다. Script는 각 실제 detection에 `frame_id`와 `captured_ms`를 붙인 뒤 TODO와 같은 `filter_detections` 계약을 통과시켜 JSON에 저장합니다. 원본 크기, class name mapping, threshold, N, latency를 확인하며 overlay 한 장만으로 stream 성능을 판단하지 않습니다.

### USB 웹캠 — 촬영하면서 결과 보기

**Jetson에 연결된 USB 웹캠을 로컬 사용자 한 명만 실행합니다.** 원격 사용자는 같은 카메라를 중복 실행하지 않고 로컬 화면을 함께 봅니다. 앞서 실행한 MediaPipe·ROS 카메라 노드 등 카메라를 사용하는 프로그램은 먼저 종료하세요.

로컬 Jetson 화면에서 사용하는 **실습 컨테이너 내부 터미널** 에 아래 명령을 붙여 넣습니다. 현재 폴더와 관계없이 실행할 수 있도록 절대 경로를 사용합니다. `--camera-index 0`은 `/dev/video0`이며, 다른 장치로 연결되었다면 번호를 바꿉니다.

```bash
bash
python /workspace/09.Pseudo-VLA/scripts/live_yolo.py --task detect --model /workspace/Data/models/yolo26n.pt --source camera --camera-index 0 --preview --max-frames 300
```

**입력 → 처리 → 화면:** USB 웹캠 프레임 → YOLO `detect` 추론 → 사각형 상자·객체 이름·confidence(검출 점수)를 원본 영상 위에 표시합니다. 카메라 앞에 사람이나 모델이 학습한 물체를 놓고 거리·조명을 바꾸며 결과를 비교하세요. 모든 물체 종류를 인식하는 것은 아니며, 검출이 없으면 원본 영상만 표시될 수 있습니다. 좌표는 좌우 반전하지 않은 입력 영상 기준입니다.

- **종료:** 결과 창에 포커스를 두고 `q` 또는 `Esc`를 누릅니다. 터미널에서 `Ctrl+C`로 끝낼 수도 있습니다. 최대 300프레임을 처리하면 자동 종료하며, 300초를 뜻하지 않습니다.
- **모델 선택:** 검출은 `yolo26n.pt`, 뒤 Pose 실습은 `yolo26n-pose.pt`를 사용합니다. `yolo26n-seg.pt`는 07 선택 자습용입니다. task와 모델이 다르면 오류로 안내합니다.
- **저장:** 종료 시 마지막으로 처리가 완료된 프레임을 `Data/outputs/09.Pseudo-VLA/yolo/detect-camera-last.jpg`와 같은 이름의 `.json`에 저장합니다. 재실행하면 덮어쓰며 전체 동영상은 저장하지 않습니다. JSON의 `frame_id`, `captured_ms`, 검출 좌표·점수를 이미지와 함께 확인합니다.
- **화면·연결 문제:** 표시 창은 브라우저가 아니라 로컬 Jetson 데스크톱에 열립니다. `DISPLAY` 오류는 로컬 그래픽 터미널과 컨테이너의 화면 연결을, 카메라 열기 실패는 USB 연결·장치 번호·컨테이너 장치 접근·다른 프로그램의 점유를 확인하세요.

**종료 후 노트북에서 마지막 결과 보기:** 아래 코드를 새 코드 셀에 붙여 넣어 실행합니다. 저장된 이미지만 읽으며 카메라나 모델을 다시 실행하지 않습니다. 파일이 없다면 카메라 실행에서 최소 한 프레임이 완료되었는지 확인합니다. 이전 실행 파일일 수도 있으므로 JSON의 캡처 시각을 함께 확인하세요.

```python
from pathlib import Path
from IPython.display import Image, display
import json

result_dir = ROOT / "Data/outputs/09.Pseudo-VLA/yolo"
image_path = result_dir / "detect-camera-last.jpg"
json_path = result_dir / "detect-camera-last.json"
if image_path.is_file() and json_path.is_file():
    display(Image(filename=str(image_path)))
    print(json.dumps(json.loads(json_path.read_text()), ensure_ascii=False, indent=2))
else:
    print("먼저 위 USB 웹캠 명령으로 한 프레임 이상 처리한 뒤 종료하세요.")
```

이 예제는 제공된 실행 스크립트의 공통 처리 함수를 사용합니다. 학생 TODO 수정이 카메라 실행에 자동 반영되는 것은 아니므로, TODO 검증 셀은 별도로 완료합니다.



## 4. YOLO26n Pose — 이미지에서 사람 전체 좌표로

Detection 모델 `yolo26n.pt`는 객체별 **박스·class·confidence** 를 냅니다. Pose 모델 `yolo26n-pose.pt`는 **사람 박스·person confidence·17개 관절 좌표와 관절별 confidence** 를 함께 냅니다. 다음 06의 통합 실행은 pose를 한 번 실행해 박스와 관절을 모두 얻습니다. detection과 pose를 연속으로 두 번 실행하는 파이프라인이 아닙니다.

| 인덱스 | 이름 | 인덱스 | 이름 |
|---:|---|---:|---|
| 0 | nose | 9 / 10 | left_wrist / right_wrist |
| 1 / 2 | left_eye / right_eye | 11 / 12 | left_hip / right_hip |
| 3 / 4 | left_ear / right_ear | 13 / 14 | left_knee / right_knee |
| 5 / 6 | left_shoulder / right_shoulder | 15 / 16 | left_ankle / right_ankle |
| 7 / 8 | left_elbow / right_elbow | | |

좌우 이름은 **화면을 보는 사람 기준이 아니라 검출된 사람의 신체 좌우** 입니다. 손가락 관절은 없으므로 이 모델로 주먹을 직접 판별하지 않습니다. 코 추종과 손목 선택처럼 몸 관절로 정의 가능한 과제를 수행합니다. [Ultralytics Pose 공식 문서](https://docs.ultralytics.com/tasks/pose/)

원본 pixel 좌표 `(px,py)`는 `(px/width,py/height)`로 정규화합니다. `x`는 오른쪽, `y`는 아래로 증가합니다. 정규화 box는 `[x1,y1,x2,y2]`입니다. 사람을 못 찾으면 `person=None`, 관절을 확실히 못 찾으면 그 관절의 `xy=None`으로 표시합니다. 좌표가 없는 관절을 `(0,0)`으로 대신 채우면 왼쪽 위를 실제 목표로 오인할 수 있습니다.

**전달 원칙:** 17개 이름을 모두 유지합니다. 선택한 사람의 박스와 모든 관절 정보를 통째로 다음 LLM 단계에 넘기며, 코 하나만 미리 잘라 보내지 않습니다. 모델은 JSON에 주어진 관절 중 자연어 지시의 대상을 선택합니다.


In [ ]:
# 모델·카메라·ROS가 필요 없는 독립 fixture입니다.
from copy import deepcopy
COCO_KEYPOINTS = ["nose", "left_eye", "right_eye", "left_ear", "right_ear", "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist", "right_wrist", "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle"]
def fixture_person(bbox, confidence, nose=(0.5, 0.3)):
    joints = {name: {"xy": [0.5, 0.5], "confidence": 0.9} for name in COCO_KEYPOINTS}
    joints["nose"]["xy"] = list(nose)
    return {"bbox": list(bbox), "confidence": confidence, "keypoints": joints}

PEOPLE = [
    fixture_person([0.2, 0.2, 0.4, 0.7], 0.95),   # 면적 0.10: 점수는 가장 높음
    fixture_person([0.1, 0.1, 0.7, 0.9], 0.80),   # 면적 0.48: 선택할 사람
    fixture_person([0.0, 0.0, 1.0, 1.0], 0.49),   # 면적 1.00: 점수 기준 미달
]
print("예측: index 1, bbox 면적 0.48; 최고 confidence와 최대 면적은 다른 기준입니다.")


## 5. TODO — 가장 큰 사람 한 명 선택

**작성 파일:** [exercises/pose_selection.py](exercises/pose_selection.py)의 `select_largest_person(people, *, min_conf=0.5)` **하나만** 구현합니다. YOLO 모델 호출, 관절 판정, LLM 요청, ROS 발행은 제공 코드의 역할입니다.

**입력:** 사람 dict의 list 또는 tuple입니다. 각 사람은 `bbox: [x1,y1,x2,y2]`, `confidence: float`, `keypoints: {관절 이름: {xy: [x,y] 또는 None, confidence: float}}`를 갖습니다. bbox와 좌표는 0–1 정규화 값입니다. 이 함수에서 사용하는 선택 근거는 사람의 **bbox와 person confidence** 이며, 코의 신뢰도로 사람을 다시 선택하지 않습니다. 관절 유효성은 다음 06의 행동 승인에서 검사합니다.

**출력:** 기준을 만족하는 가장 큰 사람의 **깊은 복사 dict** 또는 `None`입니다. 반환하는 것은 관절 하나가 아니라 전체 사람 record입니다. 입력 순서·값을 바꾸지 않고 17개 관절과 누락 상태를 그대로 보존합니다.

1. person confidence가 `min_conf` 이상인 후보만 남깁니다. 경계 `0.5`는 포함합니다.
2. bbox는 bool이 아닌 유한수 네 개이며 `0≤x1<x2≤1`, `0≤y1<y2≤1`이어야 합니다. 뒤집힘·영역 밖·면적 0·NaN·Inf 후보는 무시합니다.
3. 면적 `(x2-x1)*(y2-y1)`이 최대인 후보를 고릅니다. **면적이 같으면 입력에서 먼저 나온 후보** 를 유지합니다. 최고 confidence 순 정렬과 혼동하지 않습니다.
4. 유효 후보가 없으면 `None`을 반환합니다. 있으면 내부 관절 dict/list까지 깊게 복사합니다.

위 fixture는 0번 점수 0.95·면적 0.10, 1번 점수 0.80·면적 0.48, 2번 점수 0.49·면적 1.00입니다. 기본 경계에서는 **1번** 이 정답입니다. 검증 셀은 입력 변조와 얕은 복사까지 확인합니다.

**구현 순서 힌트:** 최대 면적과 선택된 후보를 별도로 초기화 → 각 후보를 검사 → 엄격히 더 큰 면적일 때만 교체 → 마지막에 깊은 복사. 제일 큰 사람이 가려져 코가 안 보이면 이 단계에서 다른 작은 사람으로 몰래 바꾸지 않습니다.

**설정값 오류:** `min_conf` 자체가 bool·문자열·NaN·Inf·0–1 밖이면 `ValueError`로 알립니다. 후보 하나의 오류를 건너뛰는 처리와 잘못된 설정을 구분합니다. `people`은 이 과제에서 list/tuple로 전달합니다.


In [ ]:
# 학생 파일을 저장한 뒤 이 셀을 다시 실행하세요.
student_pose = load_file("student_pose_selection", "exercises/pose_selection.py")
def check_pose_selection(select):
    before = deepcopy(PEOPLE)
    result = select(PEOPLE, min_conf=0.5)
    assert result == PEOPLE[1], "confidence를 먼저 거른 다음 bbox 면적이 가장 큰 사람을 고르세요."
    assert PEOPLE == before, "입력을 수정하면 안 됩니다."
    result["keypoints"]["nose"]["xy"][0] = 0.123
    assert PEOPLE == before, "반환값 내부 keypoints도 입력과 공유하지 않아야 합니다(deep copy)."
    assert select([], min_conf=0.5) is None
    assert select([PEOPLE[2]], min_conf=0.5) is None
    boundary = fixture_person([0.0, 0.0, 1.0, 1.0], 0.5)
    assert select([boundary], min_conf=0.5) == boundary, "threshold와 같은 점수는 포함합니다."
    tie = deepcopy(PEOPLE[1]); tie["keypoints"]["nose"]["xy"] = [0.2, 0.2]
    assert select([PEOPLE[1], tie], min_conf=0.5) == PEOPLE[1], "동일 면적은 입력에서 먼저 나온 사람입니다."
    malformed = []
    for bbox in ([0.5, 0.1, 0.4, 0.9], [0.0, 0.0, 0.0, 1.0], [-0.1, 0, 1, 1], [True, 0, 1, 1], [0, 0, float('nan'), 1]):
        bad = deepcopy(PEOPLE[1]); bad["bbox"] = bbox; malformed.append(bad)
    for confidence in (True, float('inf'), -0.1, 1.1):
        bad = deepcopy(PEOPLE[1]); bad["confidence"] = confidence; malformed.append(bad)
    assert select(malformed + [PEOPLE[0]], min_conf=0.5) == PEOPLE[0], "손상된 후보를 제외하고 유효한 후보를 선택합니다."
    assert select(malformed, min_conf=0.5) is None
    for bad_threshold in (True, "0.5", -0.1, 1.1, float("nan"), float("inf")):
        try:
            select(PEOPLE, min_conf=bad_threshold)
        except ValueError:
            pass
        else:
            raise AssertionError("잘못된 min_conf는 ValueError여야 합니다.")
    print("PASS: 최대 면적·신뢰도 경계·빈 입력·동률·잘못된 후보·깊은 복사")
check_pose_selection(student_pose.select_largest_person)


### 참고 정답 검증

학생 검증이 실패해도 정답을 먼저 읽고 원인을 비교할 수 있습니다. 아래는 동일 검증을 독립적인 참고 구현에 적용합니다. 학생 파일에 반영한 뒤 위 학생 검증을 다시 실행해야 합니다.


In [ ]:
reference_pose = load_file("reference_pose_selection", "solutions/pose_selection.py")
check_pose_selection(reference_pose.select_largest_person)


### 정답 해설 — 신뢰도 필터와 면적 선택을 분리하기

**정답 파일:** [solutions/pose_selection.py](solutions/pose_selection.py)

가장 먼저 신뢰도를 검사하는 이유는 점수가 낮은 거대한 오검출을 목표로 삼지 않기 위해서입니다. confidence가 0.49인 세 번째 사람은 bbox 면적을 비교하기 전에 제외합니다. `>=` 조건이므로 점수 0.5는 통과합니다. 숫자로 취급되는 Python bool, 유한하지 않은 NaN·Inf, 0–1 밖 점수는 유효한 검출 점수로 받지 않습니다.

bbox를 통과한 첫 번째 사람의 면적은 `(0.4-0.2)*(0.7-0.2)=0.10`입니다. 두 번째는 `(0.7-0.1)*(0.9-0.1)=0.48`이므로 갱신합니다. 갱신 조건을 `>`로 두면 동률에서는 처음 선택한 사람을 유지합니다. `>=`를 쓰면 동률에서 뒤 사람으로 바뀌므로 여기의 계약과 달라집니다.

반환값은 새 dict만 만드는 얕은 복사로 충분하지 않습니다. `keypoints['nose']['xy']`가 입력과 공유되면 화면 표시를 위해 좌표를 바꾸는 작업이 입력 기록까지 변경합니다. `copy.deepcopy`로 내부 list/dict도 분리합니다. 유효한 후보가 없다는 결과는 `None`이며 중앙에 가짜 사람을 만들지 않습니다.

**이 선택은 인물 추적이 아닙니다.** 사람이 가까이 오거나 자세로 박스가 바뀌면 다음 프레임에서 선택 대상이 전환될 수 있습니다. 박스가 가장 큰 사람과 사람의 실제 거리·중요도가 항상 일치하는 것도 아닙니다. 다음 06에서는 선택 박스를 화면에 표시해 바뀌는 이유를 관찰합니다.

<!-- pose-selection-answer -->
<details>
<summary>참고 정답 전체 코드 — 단계별 주석 포함</summary>

[실제 참고 파일](solutions/pose_selection.py)의 독립 구현입니다. 이 Markdown 코드는 자동 실행되지 않습니다. 학생 파일의 함수 시그니처를 유지하고 필요한 보조 함수도 함께 작성하세요.

```python
"""참고 정답: 신뢰도 검사 → 유효 박스 → 가장 큰 면적 → 독립 복사.

1. confidence가 높은 사람을 고르는 과제가 아닙니다. 임계값을 먼저 통과시킨 뒤
   통과한 후보의 면적을 비교합니다. 따라서 큰 저신뢰도 박스가 선택되지 않습니다.
2. 정규화 박스는 0~1의 유한수 네 개이고 x1<x2, y1<y2여야 합니다.
   뒤집힌 좌표를 abs()로 고치면 잘못된 검출을 유효한 박스로 오인하게 됩니다.
3. 넓이가 정확히 같은 경우 처음 사람을 유지하도록 비교는 >로 합니다.
   이것은 동일인 추적이 아닙니다. 다음 프레임에서는 다른 사람이 최대일 수 있습니다.
4. deepcopy가 없으면 호출자가 원래 후보/관절을 수정할 때 선택 결과도 바뀝니다.
   프레임 검증에서 입력 스냅샷을 보존하기 위해 깊은 복사를 반환합니다.
5. 후보 하나가 잘못되어도 나머지 후보를 검사합니다. 임계값 오류는 설정 오류이므로
   후보 건너뛰기와 구분하여 ValueError로 알립니다.

이 정답은 제공 ROS 구현을 import하지 않는 독립 학습용 코드입니다.
"""
from copy import deepcopy
import math

def number(value):
    # Python의 bool은 int 하위형이므로 먼저 거부해야 True=1로 처리되지 않습니다.
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError('finite number required')
    return float(value)


def unit_number(value):
    value = number(value)
    if not 0 <= value <= 1:
        raise ValueError('normalized value outside [0,1]')
    return value

def select_largest_person(people, *, min_conf=0.5):
    """Confidence filter BEFORE area comparison; first candidate wins a tie.

Invalid boxes/confidence are ignored. Returned data is independent of the input
buffer. Keypoints are normalized separately by pose_observation, not this task.
"""
    threshold = unit_number(min_conf)
    selected, largest = None, -1.0
    for person in people:
        try:
            score = unit_number(person['confidence'])
            box = person['bbox']
            if not isinstance(box, (list, tuple)) or len(box) != 4:
                continue
            x1, y1, x2, y2 = map(unit_number, box)
            if score < threshold or x2 <= x1 or y2 <= y1:
                continue
            area = (x2 - x1) * (y2 - y1)
            if area > largest:
                selected, largest = person, area
        except (ValueError, TypeError, KeyError, OverflowError):
            continue
    return deepcopy(selected)
```

</details>



In [ ]:
chosen = student_pose.select_largest_person(PEOPLE, min_conf=0.5)
pose_fixture_output = ROOT / "Data/outputs/09.Pseudo-VLA/pose-person-fixture.json"
pose_fixture_output.parent.mkdir(parents=True, exist_ok=True)
pose_fixture_output.write_text(json.dumps({"frame_id":"pose-fixture", "captured_ms":1000, "person":chosen}, ensure_ascii=False, indent=2)+"\n")
print("학생 선택 결과 저장:", pose_fixture_output)


## 6. 실제 Pose 확인 — 17개 관절과 선택된 사람

모델은 수업 전에 `/workspace/Data/models/yolo26n-pose.pt`에 준비합니다. 이미지를 네트워크로 보내는 단계는 아직 없습니다.

**정지 사진:** 새 JupyterLab 터미널에서 실행합니다. 앞선 ROS 제어기 B는 종료하고, 사진 입력은 카메라를 점유하지 않습니다.
```bash
bash
python /workspace/09.Pseudo-VLA/scripts/live_yolo.py --task pose --model /workspace/Data/models/yolo26n-pose.pt --source /workspace/Data/images/07.Mediapipe/person_open_palm.png
```

**USB 웹캠:** 앞선 ROS 카메라 A와 detection 카메라 프로그램을 먼저 종료합니다. 카메라를 사용하는 프로그램은 하나만 실행합니다. 새 JupyterLab 터미널에서 아래 전체를 실행하고, 종료는 영상 창의 `q`/`Esc` 또는 터미널의 Ctrl+C입니다.
```bash
bash
python /workspace/09.Pseudo-VLA/scripts/live_yolo.py --task pose --model /workspace/Data/models/yolo26n-pose.pt --source camera --camera-index 0 --preview --max-frames 300
```

다수 인물·코 가림·얼굴이 영상 가장자리인 조건을 비교합니다. 종료 후 마지막 사진은 `Data/outputs/09.Pseudo-VLA/yolo/pose-camera-last.jpg`, JSON은 같은 이름의 `.json`입니다. `people` 전체 목록과 `observation.person`에 선택된 사람을 확인합니다. `selected` 박스·관절을 화면과 대조하세요. 관절 좌표는 정규화 값이고 박스·관절 confidence는 서로 다른 값입니다. 재실행하면 마지막 결과 파일을 덮어씁니다.

사진·카메라 실행은 제공된 선택 구현을 사용합니다. 자신의 TODO 성공 여부는 앞 학생 검증 셀로 판정합니다. 다음 06에서 camera → pose → 가장 큰 사람 → 전체 좌표 JSON → LLM → ROS를 연결합니다.


In [ ]:
from IPython.display import Image, display
pose_result_dir = ROOT / "Data/outputs/09.Pseudo-VLA/yolo"
pose_last_json = pose_result_dir / "pose-camera-last.json"
pose_last_image = pose_result_dir / "pose-camera-last.jpg"
if pose_last_json.is_file() and pose_last_image.is_file():
    display(Image(filename=str(pose_last_image)))
    print(json.dumps(json.loads(pose_last_json.read_text()), ensure_ascii=False, indent=2))
else:
    print("미측정: 위 실제 Pose 카메라 실습을 한 프레임 이상 처리하고 종료하세요.")


## 7. 완료 확인 — 다음 ROS 실습에 넘길 데이터

- Detection TODO와 최대 면적 사람 선택 TODO 검증을 모두 통과했습니다.
- detection의 bbox와 pose의 bbox·17개 관절 차이를 설명할 수 있습니다.
- 실제 pose 결과의 사람 선택 근거·코 좌표·코 신뢰도·누락 관절을 읽었습니다.
- `pose-person-fixture.json`과 실제 pose 마지막 JSON을 구분해 저장했습니다.
- 가장 큰 사람이 바뀌면 추종 대상도 바뀌며, 코가 유효하지 않으면 다음 제어 단계는 STOP이어야 함을 설명합니다.

다음 [06 pose·LLM·ROS](06_ros_pose_llm.ipynb)는 이미지 대신 선택된 사람의 **전체 좌표** 를 LLM에 보내 “코를 따라가” 지시를 실행합니다. segmentation은 [07 선택 자습](07_yolo_segmentation.ipynb)으로 남깁니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

[4일차 슬라이드 50쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s006) · [배포 PDF 50쪽](../슬라이드/4일차.pdf#page=50)에서 50–63쪽을 복습합니다. 자신의 TODO 검증과 실제 관측·미측정을 나누어 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습 — IoU와 NMS를 숫자로 확인하기

이 자습은 수업 핵심의 confidence filtering 다음에 수행합니다. IoU는 두 box가 겹친 정도이고, NMS는 높은 score box부터 남기면서 많이 겹친 후보를 제거합니다. 아래 fixture는 실제 model 없이 항상 같은 결과를 내므로 손계산과 코드를 비교할 수 있습니다.

진행 순서는 **교집합·합집합 손계산 → IoU 함수 실행 → NMS threshold별 생존 box 예측 → 자동 검증 → Data 저장** 입니다. NMS threshold가 커지면 제거 기준이 느슨해져 더 많은 겹친 box가 남을 수 있다는 점을 먼저 예측하세요.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 자습 1단계 — 실행 fixture

Box A와 B는 각각 넓이 16이고 교집합은 9이므로 `IoU=9/(16+16-9)=9/23≈0.3913`입니다. C는 A와 더 많이 겹칩니다. 아래 `box_iou`와 `nms_fixture`는 학습을 위한 작은 구현이며 Ultralytics 내부 NMS를 그대로 복제한 것은 아닙니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
IOU_BOXES = {
    "A": [0.0, 0.0, 4.0, 4.0],
    "B": [1.0, 1.0, 5.0, 5.0],
    "C": [0.5, 0.5, 4.5, 4.5],
}

def box_iou(left, right):
    lx1, ly1, lx2, ly2 = left
    rx1, ry1, rx2, ry2 = right
    intersection_width = max(0.0, min(lx2, rx2) - max(lx1, rx1))
    intersection_height = max(0.0, min(ly2, ry2) - max(ly1, ry1))
    intersection = intersection_width * intersection_height
    left_area = (lx2 - lx1) * (ly2 - ly1)
    right_area = (rx2 - rx1) * (ry2 - ry1)
    union = left_area + right_area - intersection
    return intersection / union if union else 0.0

NMS_CANDIDATES = [
    {"name": "A", "confidence": 0.90, "bbox_xyxy": IOU_BOXES["A"]},
    {"name": "C", "confidence": 0.85, "bbox_xyxy": IOU_BOXES["C"]},
    {"name": "B", "confidence": 0.80, "bbox_xyxy": IOU_BOXES["B"]},
]

def nms_fixture(candidates, iou_threshold):
    ordered = sorted(candidates, key=lambda item: (-item["confidence"], item["name"]))
    kept = []
    for candidate in ordered:
        if all(box_iou(candidate["bbox_xyxy"], previous["bbox_xyxy"]) <= iou_threshold for previous in kept):
            kept.append(candidate)
    return kept

print("IoU(A,B)=", round(box_iou(IOU_BOXES["A"], IOU_BOXES["B"]), 4))
print("IoU(A,C)=", round(box_iou(IOU_BOXES["A"], IOU_BOXES["C"]), 4))


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 자습 TODO와 예상 결과

1. 코드를 실행하기 전에 threshold `0.30`, `0.50`, `0.70`에서 남을 box 이름을 종이에 적습니다.
2. 기본 예상 결과는 `0.30→[A]`, `0.50→[A,B]`, `0.70→[A,C,B]`입니다.
3. **TODO:** 다음 셀에서 `C`의 좌표와 confidence를 모두 채우되 둘 중 하나만 원래값에서 바꿉니다. 바꾸지 않는 값은 원래 좌표 `[0.5,0.5,4.5,4.5]` 또는 confidence `0.85`를 복사합니다. 세 threshold의 새 결과와 이유도 채웁니다. 기본 fixture 자체는 바꾸지 않습니다.
4. 검증 뒤 JSON을 `Data/outputs/09.Pseudo-VLA/self-study/01-iou-nms.json`에서 엽니다. TODO 전에는 `completed=false`, 정답을 채우면 `true`입니다.

**작성 변수와 출력 형식:** 다음 셀의 `MY_C_BOX`는 NaN·무한대를 제외한 유효한 pixel xyxy 네 숫자의 list, `MY_C_CONFIDENCE`는 0~1 유한수입니다. 원래값과 비교하여 좌표 또는 점수 중 정확히 하나만 바꾸고 두 변수는 모두 채웁니다. `MY_PREDICTED_NMS`는 문자열 key "0.30"/"0.50"/"0.70" 각각에 남을 이름의 순서 있는 list를 넣는 dict이며 `MY_NMS_EXPLANATION`은 이유 한 문장입니다. 예측은 C만 바꾼 fixture를 높은 confidence 순서로 처리하면서 이미 남긴 box와의 IoU가 threshold를 초과하면 제거하는 순서로 구합니다. 아래 검증은 세 예상 목록과 실제 목록, 한 조건만 변경했는지, 설명이 비어 있지 않은지를 확인합니다. 입력 목록·기본 fixture는 수정하지 않습니다.

### 정답을 해석하는 순서와 오류 복구

점수가 높은 후보부터 하나를 남기고, 이미 남긴 모든 box와 새 후보의 IoU를 비교합니다. IoU가 threshold와 **같으면 유지** 하고 **초과하면 제거** 합니다. threshold 0.30에서 A만 남는다는 것은 B·C가 검출되지 않았다는 뜻이 아니라 겹침 제거 결과입니다. C의 좌표를 바꿨다면 면적·교집합이, 점수를 바꿨다면 처리 순서가 달라지는지 설명하세요. 이 자습은 고전적인 NMS 계산을 이해하는 fixture이며 실제 YOLO 실행기의 내부 처리를 그대로 재현하는 과제가 아닙니다.

| 증상 | 확인할 내용과 조치 |
|---|---|
| `NameError` | 앞의 IoU/NMS fixture 셀부터 실행. 다른 노트북의 변수는 공유되지 않음 |
| 유한수·좌표 assertion | 네 값 모두 숫자이며 NaN·무한대가 아닌지, x1<x2·y1<y2인지 확인 |
| 한 조건 변경 assertion | 좌표와 점수를 동시에 바꾸지 말고 한쪽을 원래값으로 복원 |
| 예상 목록 불일치 | 점수 내림차순 → 이미 남긴 각 box와 IoU → 초과 여부 순으로 손계산 |
| 저장 오류 | 출력된 경로·권한·저장 공간 확인. 검증 실패를 숨기고 completed를 직접 바꾸지 않음 |

예시에 없는 오류는 [공통 자습 오류 안내](../자습_오류해결.md)의 로그 수집·문의 양식을 사용합니다. TODO가 남아 completed=false인 기록은 오류가 아니라 미완료 기록입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
MY_C_BOX = None               # TODO: 좌표를 바꾸거나 원래 [0.5, 0.5, 4.5, 4.5] 복사
MY_C_CONFIDENCE = None        # TODO: 값을 바꾸거나 원래 0.85 복사
MY_PREDICTED_NMS = {
    "0.30": None,             # TODO 예: ["A"]
    "0.50": None,
    "0.70": None,
}
MY_NMS_EXPLANATION = ""       # TODO: 어떤 IoU 비교가 결과를 바꿨는지 한 문장
print("현재 자습 답:", MY_C_BOX, MY_C_CONFIDENCE, MY_PREDICTED_NMS)


In [ ]:
# 구분: 수업 후 복습·자습 셀
import math

thresholds = (0.30, 0.50, 0.70)
nms_records = [
    {"threshold": threshold, "kept": [item["name"] for item in nms_fixture(NMS_CANDIDATES, threshold)]}
    for threshold in thresholds
]
assert round(box_iou(IOU_BOXES["A"], IOU_BOXES["B"]), 4) == round(9 / 23, 4)
assert [item["kept"] for item in nms_records] == [["A"], ["A", "B"], ["A", "C", "B"]]

experiment_completed = False
experiment_record = None
if (
    MY_C_BOX is None or MY_C_CONFIDENCE is None
    or any(value is None for value in MY_PREDICTED_NMS.values())
    or not MY_NMS_EXPLANATION.strip()
):
    print("TODO가 남았습니다. C 변경값, 세 예측과 설명을 채우고 다시 실행하세요.")
else:
    assert isinstance(MY_C_BOX, (list, tuple)) and len(MY_C_BOX) == 4
    assert all(isinstance(value, (int, float)) and not isinstance(value, bool) for value in MY_C_BOX)
    assert all(math.isfinite(value) for value in MY_C_BOX), "좌표는 NaN·무한대가 아닌 유한수여야 합니다"
    assert 0 <= MY_C_BOX[0] < MY_C_BOX[2] and 0 <= MY_C_BOX[1] < MY_C_BOX[3]
    assert isinstance(MY_C_CONFIDENCE, (int, float)) and not isinstance(MY_C_CONFIDENCE, bool)
    assert math.isfinite(MY_C_CONFIDENCE) and 0 <= MY_C_CONFIDENCE <= 1
    box_changed = list(MY_C_BOX) != IOU_BOXES["C"]
    confidence_changed = float(MY_C_CONFIDENCE) != 0.85
    assert box_changed ^ confidence_changed, "좌표와 confidence 중 정확히 하나만 원래값에서 바꾸세요"
    changed_candidates = [
        ({**item, "bbox_xyxy": list(MY_C_BOX), "confidence": float(MY_C_CONFIDENCE)}
         if item["name"] == "C" else dict(item))
        for item in NMS_CANDIDATES
    ]
    actual_prediction = {
        f"{threshold:.2f}": [item["name"] for item in nms_fixture(changed_candidates, threshold)]
        for threshold in thresholds
    }
    assert MY_PREDICTED_NMS == actual_prediction
    experiment_completed = True
    experiment_record = {
        "c_box": list(MY_C_BOX), "c_confidence": float(MY_C_CONFIDENCE),
        "actual_nms": actual_prediction, "explanation": MY_NMS_EXPLANATION.strip(),
    }
    print("PASS: 바꾼 C에 대한 세 NMS 예측이 실제 계산과 같습니다.")

iou_output = ROOT / "Data/outputs/09.Pseudo-VLA/self-study/01-iou-nms.json"
iou_output.parent.mkdir(parents=True, exist_ok=True)
iou_output.write_text(json.dumps({
    "iou_a_b": box_iou(IOU_BOXES["A"], IOU_BOXES["B"]),
    "iou_a_c": box_iou(IOU_BOXES["A"], IOU_BOXES["C"]),
    "baseline_nms": nms_records,
    "learner_experiment": experiment_record,
    "completed": experiment_completed,
}, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print("PASS baseline:", nms_records)
print("saved:", iou_output)
